# Lecture 06: Neural Networks — Exercises

Companion to the note [Noter små](../Noter%20små.md).

Practise the neural-network part of the course the way the exam asks it: forward passes with ReLU and sigmoid units, perceptron and gradient-descent weight updates, and one full step of backpropagation by hand. Then implement layers, backpropagation (checked against finite differences) and training loops in NumPy.

**16 exercises** · 🧠 Concept ×4 · ✍️ By hand ×6 · 💻 Code ×6

**How to use this notebook**
- Difficulty: ★☆☆ warm-up · ★★☆ standard · ★★★ challenge.
- ✍️ *By hand* exercises are meant for pen and paper (or a markdown cell). 💻 *Code* exercises have a cell for you to fill in.
- Every exercise has a folded **💡 Hint** and **✅ Solution**. Click to unfold, but try first!
- The notebook runs top to bottom as-is; `check(...)` lines print ⏳ until you have an answer, then ✅ or ❌.


In [ ]:
import numpy as np
np.random.seed(0)

# --- helper: compares your answer with a reference, prints ✅/❌, never raises ---
import numpy as _np
def check(name, got, expected, tol=1e-6):
    if got is None or got is Ellipsis:
        print(f"⏳ {name}: not attempted yet")
        return
    try:
        ok = bool(_np.allclose(_np.asarray(got, dtype=float), _np.asarray(expected, dtype=float), atol=tol, rtol=1e-4))
    except Exception:
        try:
            ok = bool(got == expected)
        except Exception:
            ok = False
    print(("✅ " if ok else "❌ ") + name + ("" if ok else f"  (got {got!r})"))

from sklearn.neural_network import MLPRegressor, MLPClassifier
from sklearn.linear_model import Perceptron, LogisticRegression
from sklearn.datasets import make_moons, make_blobs
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings('ignore')
rng = np.random.default_rng(6)

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def relu(z):
    return np.maximum(0, z)

## Part A · Concepts

### Exercise 1 · Encoding categorical inputs
*🧠 Concept · ★☆☆*

You want a neural network to predict *UserAction* (reads/skips) from *Author* (known/unknown), *Thread* (new/followUp),
*Length* (long/short) and *WhereRead* (home/work).

1. Propose two different input encodings and give the number of input units for each.
2. How would you encode the output, and which output activation would you use?
3. If *Length* had three values (short/medium/long), why is the encoding short = 0, medium = 1, long = 2 not always a good idea?

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

Either one unit per binary attribute (0/1), or one unit per attribute–value combination (one-hot).

</details>

<details>
<summary><b>✅ Solution</b></summary>

1. (a) One 0/1 unit per attribute: **4 inputs** (e.g. known = 1, unknown = 0). (b) One-hot: one unit per attribute–value pair:
   **8 inputs** (Author=known, Author=unknown, …). One-hot also works for attributes with more than two values.
2. One output unit with a sigmoid, interpreted as $P(\text{reads})$ and trained with log loss (or two units with softmax).
3. An integer code imposes an order and equal spacing, and the network sees "long = 2 × medium". For ordinal attributes that can be fine;
   for nominal attributes (e.g. colours) it invents structure that is not there. One-hot avoids this.

</details>

### Exercise 2 · Why do we need non-linear activations?
*🧠 Concept · ★☆☆*

Show that a network with two layers that both use the identity activation, $\text{out} = W_2(W_1x + b_1) + b_2$, computes a function that
a single linear layer can also compute. What does this mean for deep networks without activations? What kind of function does a
network with only ReLU hidden units compute?

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

Multiply out the brackets.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$W_2(W_1x + b_1) + b_2 = (W_2W_1)x + (W_2b_1 + b_2) = Wx + b$: one linear layer. Stacking any number of linear layers is still linear,
so depth adds no expressive power without non-linear activations. With ReLU hidden units the network is **piecewise linear** (as the note
says): each ReLU switches between two linear pieces, and the pieces combine into a function with many linear regions.

</details>

### Exercise 3 · XOR and the single neuron
*🧠 Concept · ★★☆*

A single neuron with a step/sign activation outputs 1 iff $w_0 + w_1x_1 + w_2x_2 > 0$.

1. Show that no such neuron computes XOR on $\{0,1\}^2$.
2. Give a two-layer network of threshold units that computes XOR (choose the weights yourself).

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint 1</b></summary>

Write the four inequalities that the weights would have to satisfy and add pairs of them.

</details>

<details>
<summary><b>💡 Hint 2</b></summary>

XOR$(x_1,x_2)$ = OR$(x_1,x_2)$ AND NOT AND$(x_1,x_2)$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

1. We would need $w_0 \le 0$ (input 00), $w_0 + w_1 > 0$ (10), $w_0 + w_2 > 0$ (01) and $w_0 + w_1 + w_2 \le 0$ (11). Adding the middle
   two gives $2w_0 + w_1 + w_2 > 0$, while adding the first and last gives $2w_0 + w_1 + w_2 \le 0$. Contradiction: XOR is not linearly separable.
2. Hidden unit $h_1 = [x_1 + x_2 - 0.5 > 0]$ (OR), $h_2 = [x_1 + x_2 - 1.5 > 0]$ (AND), output $o = [h_1 - h_2 - 0.5 > 0]$.
   Check: 00 → (0,0) → 0; 01, 10 → (1,0) → 1; 11 → (1,1) → 0. ✔

</details>

### Exercise 4 · Vanishing gradients and dead ReLUs
*🧠 Concept · ★★☆*

1. The derivative of the sigmoid is at most $0.25$. Backpropagating through 10 sigmoid layers multiplies (at least) 10 such factors. Give an upper
   bound on that product and explain the consequence for training deep sigmoid networks.
2. ReLU has derivative 1 for positive inputs. What is the risk instead (a "dead" ReLU)?

In [ ]:
bound_10_layers = None

check('0.25^10', bound_10_layers, 0.25 ** 10)

<details>
<summary><b>💡 Hint</b></summary>

$0.25^{10} = 2^{-20}$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

1. $0.25^{10} = 2^{-20} \approx 9.5\cdot10^{-7}$. Gradients reaching the first layers are tiny, so these layers barely learn (vanishing gradient).
2. If a ReLU unit's pre-activation is negative for every training example, its derivative is 0 everywhere (the note assumes 0 also at $x=0$),
   so its incoming weights never change: the unit is dead. Causes: a too-large learning rate or a strongly negative bias. Fixes: smaller
   learning rates, careful initialisation, leaky ReLU.

```python
bound_10_layers = 0.25 ** 10
```

</details>

## Part B · By hand

### Exercise 5 · ReLU forward pass (exam style)
*✍️ By hand · ★☆☆*

Inputs *Enjoyment* and *Length* feed two hidden ReLU units (Pink, Orange) and one output *Rating* with identity activation. All biases are 0.

| weight | Enjoyment | Length |
|:--:|:--:|:--:|
| Pink | 0.5 | 1.2 |
| Orange | 2.0 | −1.5 |

Output weights: Pink → Rating $0.8$, Orange → Rating $3.0$.

Compute the Rating for observation 1: (Enjoyment, Length) $= (6, 10)$ and observation 2: $(4, 2)$.

In [ ]:
rating1 = None
rating2 = None

_W1 = np.array([[0.5, 2.0], [1.2, -1.5]]); _w2 = np.array([0.8, 3.0])
_m = MLPRegressor(hidden_layer_sizes=(2,), activation='relu', max_iter=1).fit(np.zeros((2, 2)), [0, 1])
_m.coefs_ = [_W1, _w2.reshape(2, 1)]; _m.intercepts_ = [np.zeros(2), np.zeros(1)]
_ref = _m.predict(np.array([[6., 10.], [4., 2.]]))
check('rating 1', rating1, _ref[0]); check('rating 2', rating2, _ref[1])

<details>
<summary><b>💡 Hint</b></summary>

Compute both pre-activations, apply $\max(0,\cdot)$, then take the weighted sum for the output.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Obs. 1: Pink $= 0.5\cdot6 + 1.2\cdot10 = 15$; Orange $= 2\cdot6 - 1.5\cdot10 = -3 \to \text{ReLU} = 0$. Rating $= 0.8\cdot15 + 3\cdot0 = 12$.

Obs. 2: Pink $= 2 + 2.4 = 4.4$; Orange $= 8 - 3 = 5$. Rating $= 0.8\cdot4.4 + 3\cdot5 = 3.52 + 15 = 18.52$.

In observation 1 the Orange unit is inactive, so its outgoing weight gets no gradient from this example.

```python
rating1 = 12.0
rating2 = 18.52
```

</details>

### Exercise 6 · Sigmoid hidden layer with biases
*✍️ By hand · ★★☆*

Inputs *Fat* and *Sugars*. Two sigmoid hidden units and a linear output:

* $h_1 = \sigma(0.2 + 1.0\cdot\text{Fat} - 0.4\cdot\text{Sugars})$
* $h_2 = \sigma(-1.0 + 0.5\cdot\text{Fat} + 0.3\cdot\text{Sugars})$
* $\text{Rating} = 0.5 + 2.0\,h_1 - 1.5\,h_2$

Compute the Rating for (Fat, Sugars) $= (1, 5)$ and $(0, 3)$ (4 decimals).

In [ ]:
rating_a = None
rating_b = None

_m = MLPRegressor(hidden_layer_sizes=(2,), activation='logistic', max_iter=1).fit(np.zeros((2, 2)), [0, 1])
_m.coefs_ = [np.array([[1.0, 0.5], [-0.4, 0.3]]), np.array([[2.0], [-1.5]])]; _m.intercepts_ = [np.array([0.2, -1.0]), np.array([0.5])]
_r = _m.predict(np.array([[1., 5.], [0., 3.]]))
check('rating (1,5)', rating_a, _r[0], tol=1e-3); check('rating (0,3)', rating_b, _r[1], tol=1e-3)

<details>
<summary><b>💡 Hint</b></summary>

For $(1,5)$ the pre-activations are $-0.8$ and $1.0$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$(1,5)$: $h_1 = \sigma(-0.8) = 0.3100$, $h_2 = \sigma(1.0) = 0.7311$; Rating $= 0.5 + 0.6201 - 1.0966 = 0.0235$.

$(0,3)$: $h_1 = \sigma(-1.0) = 0.2689$, $h_2 = \sigma(-0.1) = 0.4750$; Rating $= 0.5 + 0.5379 - 0.7125 = 0.3254$.

```python
rating_a = 0.0235
rating_b = 0.3254
```

</details>

### Exercise 7 · Perceptron training with the sign function
*✍️ By hand · ★★☆*

Training examples ($x_0 = 1$ is the threshold input):

| $X_1$ | $X_2$ | $T$ |
|:--:|:--:|:--:|
| 1 | 1 | 1 |
| −1 | 1 | −1 |
| 1 | −1 | 1 |
| −1 | −1 | −1 |

Initial weights $w = (w_0, w_1, w_2) = (0, 0, 0)$, activation $\text{sign}$ with $\text{sign}(0) = -1$, learning rate $\alpha = 0.25$.
Do one pass of **stochastic** gradient descent (examples one by one, in table order) with the update $w \leftarrow w + \alpha(t - o)x$.
Give $w$ after each example. Has the perceptron learned the data?

In [ ]:
w_after = None   # list of 4 weight vectors, e.g. [[w0, w1, w2], ...]

_X = np.array([[1, 1, 1], [1, -1, 1], [1, 1, -1], [1, -1, -1]], float); _T = [1, -1, 1, -1]
_w = np.zeros(3); _ref = []
for _x, _t in zip(_X, _T):
    _o = 1 if _w @ _x > 0 else -1; _w = _w + 0.25 * (_t - _o) * _x; _ref.append(_w.copy())
check('weights after each example', w_after, _ref)

<details>
<summary><b>💡 Hint</b></summary>

If the output is already correct, $t - o = 0$ and nothing changes. A mistake changes the weights by $\pm 0.5\,x$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

1. $x = (1,1,1)$: $o = \text{sign}(0) = -1$, $t = 1$, $w \leftarrow 0 + 0.25\cdot2\cdot(1,1,1) = (0.5, 0.5, 0.5)$.
2. $x = (1,-1,1)$: $w\cdot x = 0.5$, $o = 1$, $t = -1$: $w \leftarrow (0.5,0.5,0.5) - 0.5(1,-1,1) = (0, 1, 0)$.
3. $x = (1,1,-1)$: $w\cdot x = 1$, $o = 1 = t$: no change.
4. $x = (1,-1,-1)$: $w\cdot x = -1$, $o = -1 = t$: no change.

Final $w = (0, 1, 0)$, i.e. output $=\text{sign}(X_1)$, which classifies all four examples correctly.

```python
w_after = [[0.5, 0.5, 0.5], [0, 1, 0], [0, 1, 0], [0, 1, 0]]
```

</details>

### Exercise 8 · Deriving the delta rule
*✍️ By hand · ★★☆*

For a single linear unit with $E(w) = \tfrac12\sum_{i=1}^M (y_i - \sum_j x_{i,j}w_j)^2$ (the note's error function):

1. Compute $\partial E/\partial w_j$ and write the update $\Delta w_j = -\alpha\,\partial E/\partial w_j$.
2. Why does the $\tfrac12$ not change the minimiser?
3. Perform one batch update with $\alpha = 0.1$ from $w = (0, 0)$ on the data $x_1 = (1, 2), y_1 = 3$ and $x_2 = (1, -1), y_2 = 0$.

In [ ]:
w_new = None  # [w0, w1]

_X = np.array([[1, 2], [1, -1.]]); _y = np.array([3, 0.])
check('updated weights', w_new, np.zeros(2) + 0.1 * _X.T @ (_y - _X @ np.zeros(2)))

<details>
<summary><b>💡 Hint</b></summary>

$\partial/\partial w_j\ \tfrac12(y_i - o_i)^2 = -(y_i - o_i)x_{i,j}$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

1. $\partial E/\partial w_j = -\sum_i (y_i - o_i)x_{i,j}$, so $\Delta w_j = \alpha\sum_i (y_i - o_i)x_{i,j}$: increase $w_j$ when the output is too small
   and $x_{i,j} > 0$.
2. Scaling a function by a positive constant does not move its minimum; the $\tfrac12$ only cancels the 2 from differentiating the square.
3. Outputs are 0, errors $(3, 0)$. $\Delta w = 0.1\,(3\cdot(1,2) + 0\cdot(1,-1)) = (0.3, 0.6)$, so $w = (0.3, 0.6)$.

```python
w_new = [0.3, 0.6]
```

</details>

### Exercise 9 · One step of backpropagation
*✍️ By hand · ★★★*

A network with inputs $x = (x_1, x_2) = (1, 2)$, two sigmoid hidden units and one sigmoid output, no biases.
Weights input → hidden (row = input, column = hidden unit) and hidden → output:
$$W = \begin{pmatrix}0.3 & -0.2\\ 0.1 & 0.4\end{pmatrix},\qquad v = (0.5, -0.6).$$
Target $y = 1$, error $E = \tfrac12(y - o)^2$, learning rate $\eta = 0.5$.

Compute (4 decimals): the hidden outputs $h$, the output $o$, the error $E$, the output error term
$\delta_o = (o - y)\,o(1-o)$, the updated $v$, the hidden error terms $\delta_{h_j} = \delta_o v_j h_j(1-h_j)$ and the updated $W$.

In [ ]:
h_bp = None
o_bp = None
E_bp = None
delta_o = None
v_new = None
delta_h = None
W_new = None

_x = np.array([1., 2.]); _W = np.array([[0.3, -0.2], [0.1, 0.4]]); _v = np.array([0.5, -0.6])
def _E(W, v): return 0.5 * (1 - 1 / (1 + np.exp(-(1 / (1 + np.exp(-_x @ W))) @ v))) ** 2
_eps = 1e-6
_gv = np.array([(_E(_W, _v + _eps * e) - _E(_W, _v - _eps * e)) / (2 * _eps) for e in np.eye(2)])
_gW = np.array([[(_E(_W + _eps * np.outer(a, b), _v) - _E(_W - _eps * np.outer(a, b), _v)) / (2 * _eps) for b in np.eye(2)] for a in np.eye(2)])
check('E', E_bp, _E(_W, _v), tol=1e-4)
check('updated v (vs finite differences)', v_new, _v - 0.5 * _gv, tol=1e-4)
check('updated W (vs finite differences)', W_new, _W - 0.5 * _gW, tol=1e-4)

<details>
<summary><b>💡 Hint 1</b></summary>

Pre-activations: $a = x^\top W = (0.5, 0.6)$, so $h = (\sigma(0.5), \sigma(0.6))$.

</details>

<details>
<summary><b>💡 Hint 2</b></summary>

Weight updates: $v_j \leftarrow v_j - \eta\,\delta_o h_j$ and $W_{ij} \leftarrow W_{ij} - \eta\,\delta_{h_j} x_i$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

* $h = (\sigma(0.5), \sigma(0.6)) = (0.6225, 0.6457)$; output pre-activation $0.5\cdot0.6225 - 0.6\cdot0.6457 = -0.0761$, $o = 0.4810$.
* $E = \tfrac12(1 - 0.4810)^2 = 0.1347$.
* $\delta_o = (0.4810 - 1)\cdot0.4810\cdot0.5190 = -0.1296$.
* $v_{new} = v - 0.5\cdot(-0.1296)\,h = (0.5403, -0.5582)$.
* $\delta_h = \delta_o\,v \odot h \odot (1-h) = (-0.0152, 0.0178)$ (using the **old** $v$).
* $W_{new} = W - 0.5\,x\,\delta_h^\top = \begin{pmatrix}0.3076 & -0.2089\\ 0.1152 & 0.3822\end{pmatrix}$.

The check compares with gradients from finite differences, the standard way to test a backprop implementation.

```python
h_bp = np.array([0.6225, 0.6457]); o_bp = 0.4810; E_bp = 0.1347; delta_o = -0.1296
v_new = np.array([0.5403, -0.5582]); delta_h = np.array([-0.0152, 0.0178])
W_new = np.array([[0.3076, -0.2089], [0.1152, 0.3822]])
```

</details>

### Exercise 10 · Counting parameters
*✍️ By hand · ★☆☆*

A fully connected network has 4 inputs, one hidden layer of 5 units and 3 outputs, with a bias for every hidden and output unit.
How many weights (including biases) does it have? How many with a second hidden layer of 5 units?

In [ ]:
n_params_1 = None
n_params_2 = None

_m1 = MLPClassifier(hidden_layer_sizes=(5,), max_iter=1).fit(rng.normal(size=(9, 4)), np.arange(9) % 3)
_m2 = MLPClassifier(hidden_layer_sizes=(5, 5), max_iter=1).fit(rng.normal(size=(9, 4)), np.arange(9) % 3)
_count = lambda m: sum(c.size for c in m.coefs_) + sum(b.size for b in m.intercepts_)
check('4-5-3', n_params_1, _count(_m1)); check('4-5-5-3', n_params_2, _count(_m2))

<details>
<summary><b>💡 Hint</b></summary>

A layer from $n$ to $m$ units has $n\cdot m$ weights plus $m$ biases (the note's extra constant input 1).

</details>

<details>
<summary><b>✅ Solution</b></summary>

4-5-3: $(4\cdot5 + 5) + (5\cdot3 + 3) = 25 + 18 = 43$. 4-5-5-3: $25 + (5\cdot5 + 5) + 18 = 73$.

```python
n_params_1 = 43
n_params_2 = 73
```

</details>

## Part C · Code

### Exercise 11 · A dense layer and a forward pass
*💻 Code · ★☆☆*

Implement `dense(inp, W, b, phi)` computing $\text{out}[j] = \phi(\sum_k \text{in}[k]\,w[k,j] + b[j])$ for a batch of inputs (rows), and
`mlp_forward(X, layers)` where `layers` is a list of `(W, b, phi)`. Use it to reproduce the ReLU exam network on both observations.

In [ ]:
def dense(inp, W, b, phi):
    # TODO
    return None

def mlp_forward(X, layers):
    # TODO
    return None

identity = lambda z: z
layers_exam = [(np.array([[0.5, 2.0], [1.2, -1.5]]), np.zeros(2), relu),
               (np.array([[0.8], [3.0]]), np.zeros(1), identity)]
ratings = mlp_forward(np.array([[6., 10.], [4., 2.]]), layers_exam)

check('ratings', None if ratings is None else np.ravel(ratings), [12.0, 18.52])

<details>
<summary><b>💡 Hint</b></summary>

With rows as examples: `phi(inp @ W + b)`.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Writing the layer as a matrix product processes the whole batch at once. The bias is the weight of the constant input 1, so
`inp @ W + b` is the same as appending a column of ones to `inp` and a row `b` to `W`.

```python
def dense(inp, W, b, phi):
    return phi(inp @ W + b)

def mlp_forward(X, layers):
    out = X
    for W, b, phi in layers:
        out = dense(out, W, b, phi)
    return out

identity = lambda z: z
layers_exam = [(np.array([[0.5, 2.0], [1.2, -1.5]]), np.zeros(2), relu),
               (np.array([[0.8], [3.0]]), np.zeros(1), identity)]
ratings = mlp_forward(np.array([[6., 10.], [4., 2.]]), layers_exam)
```

</details>

### Exercise 12 · Backpropagation for a one-hidden-layer network
*💻 Code · ★★★*

Network: sigmoid hidden layer, **linear** output, loss $E = \tfrac12\sum_i\lVert o_i - y_i\rVert^2$ over a batch.
Implement `loss_and_grads(X, Y, W1, b1, W2, b2)` returning `(E, dW1, db1, dW2, db2)`. The check compares your gradients with finite differences.

In [ ]:
def loss_and_grads(X, Y, W1, b1, W2, b2):
    # forward: H = sigmoid(X W1 + b1); O = H W2 + b2
    # backward: dO = O - Y; dW2 = H^T dO; ...
    return None

Xg = rng.normal(size=(5, 3)); Yg = rng.normal(size=(5, 2))
W1g, b1g = rng.normal(size=(3, 4)), rng.normal(size=4)
W2g, b2g = rng.normal(size=(4, 2)), rng.normal(size=2)
grads = loss_and_grads(Xg, Yg, W1g, b1g, W2g, b2g)

def _num_grad(f, P, eps=1e-6):
    G = np.zeros_like(P)
    for i in np.ndindex(P.shape):
        old = P[i]; P[i] = old + eps; fp = f(); P[i] = old - eps; fm = f(); P[i] = old; G[i] = (fp - fm) / (2 * eps)
    return G
_f = lambda: 0.5 * np.sum((sigmoid(Xg @ W1g + b1g) @ W2g + b2g - Yg) ** 2)
if grads is None:
    check('gradients match finite differences', None, True)
else:
    _ok = all(np.allclose(g, _num_grad(_f, P), atol=1e-5) for g, P in zip(grads[1:], [W1g, b1g, W2g, b2g]))
    check('gradients match finite differences', _ok, True)

<details>
<summary><b>💡 Hint 1</b></summary>

Output layer: $\delta_O = O - Y$, $\partial E/\partial W_2 = H^\top\delta_O$, $\partial E/\partial b_2 = \sum_i \delta_{O,i}$.

</details>

<details>
<summary><b>💡 Hint 2</b></summary>

Hidden layer: $\delta_H = (\delta_O W_2^\top)\odot H\odot(1-H)$, then $\partial E/\partial W_1 = X^\top\delta_H$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Backpropagation is the chain rule applied layer by layer from the output backwards. Each layer's error term is the next layer's error
term pulled back through the weights ($\delta W^\top$) and multiplied by the local derivative of the activation. The bias gradient is
the error term itself (its "input" is the constant 1), summed over the batch.

```python
def loss_and_grads(X, Y, W1, b1, W2, b2):
    H = sigmoid(X @ W1 + b1)
    O = H @ W2 + b2
    E = 0.5 * np.sum((O - Y) ** 2)
    dO = O - Y
    dW2 = H.T @ dO; db2 = dO.sum(0)
    dH = (dO @ W2.T) * H * (1 - H)
    dW1 = X.T @ dH; db1 = dH.sum(0)
    return E, dW1, db1, dW2, db2

Xg = rng.normal(size=(5, 3)); Yg = rng.normal(size=(5, 2))
W1g, b1g = rng.normal(size=(3, 4)), rng.normal(size=4)
W2g, b2g = rng.normal(size=(4, 2)), rng.normal(size=2)
grads = loss_and_grads(Xg, Yg, W1g, b1g, W2g, b2g)
```

</details>

### Exercise 13 · Learning XOR
*💻 Code · ★★★*

Train a 2-4-1 network (sigmoid hidden units, linear output) on XOR with full-batch gradient descent using your `loss_and_grads`
(learning rate 0.2, 5000 epochs, weights from `np.random.default_rng(1).normal(size=...)`). Store the final loss in `xor_loss` and the
rounded predictions in `xor_pred`. Compare with what a single neuron can achieve (Exercise 3).

In [ ]:
X_xor = np.array([[0, 0], [0, 1], [1, 0], [1, 1]], float)
Y_xor = np.array([[0], [1], [1], [0]], float)
xor_loss = None
xor_pred = None

check('XOR loss small', None if xor_loss is None else xor_loss < 0.01, True)
check('XOR predictions', xor_pred, [0, 1, 1, 0])

<details>
<summary><b>💡 Hint</b></summary>

Loop: compute `E, dW1, db1, dW2, db2`, then `W1 -= lr * dW1` etc. The loss is a *sum* over examples, so too large a step (try 0.5) diverges. If it gets stuck, try another seed: XOR has local minima.

</details>

<details>
<summary><b>✅ Solution</b></summary>

The hidden layer learns two features (roughly OR and AND, as in the hand-built network) that make XOR linearly separable for the output
unit. The best linear model can only predict 0.5 everywhere (loss 0.5).

```python
X_xor = np.array([[0, 0], [0, 1], [1, 0], [1, 1]], float)
Y_xor = np.array([[0], [1], [1], [0]], float)
r = np.random.default_rng(1)
W1, b1 = r.normal(size=(2, 4)), r.normal(size=4)
W2, b2 = r.normal(size=(4, 1)), r.normal(size=1)
for epoch in range(5000):
    E, dW1, db1, dW2, db2 = loss_and_grads(X_xor, Y_xor, W1, b1, W2, b2)
    W1 -= 0.2 * dW1; b1 -= 0.2 * db1; W2 -= 0.2 * dW2; b2 -= 0.2 * db2
xor_loss = loss_and_grads(X_xor, Y_xor, W1, b1, W2, b2)[0]
xor_pred = np.round(sigmoid(X_xor @ W1 + b1) @ W2 + b2).ravel().astype(int).tolist()
print(xor_loss, xor_pred)
```

</details>

### Exercise 14 · The perceptron learning rule
*💻 Code · ★★☆*

Implement `perceptron_train(X, t, alpha, epochs)` with $t \in \{-1, 1\}$, sign activation ($\text{sign}(0) = -1$), bias input $x_0 = 1$ and the
stochastic update $w \leftarrow w + \alpha(t - o)x$. Train on the separable blobs below and compare the training accuracy with
`sklearn.linear_model.Perceptron`.

In [ ]:
X_p, y_p = make_blobs(n_samples=100, centers=[[-2, -2], [2, 2]], cluster_std=1.0, random_state=0)
t_p = np.where(y_p == 1, 1, -1)

def perceptron_train(X, t, alpha=0.25, epochs=20):
    # TODO: return w = [w0, w1, w2]
    return None

w_p = perceptron_train(X_p, t_p)
acc_mine = None if w_p is None else np.mean(np.where(np.c_[np.ones(len(X_p)), X_p] @ w_p > 0, 1, -1) == t_p)

check('my perceptron accuracy = sklearn', acc_mine, Perceptron(random_state=0).fit(X_p, t_p).score(X_p, t_p))

<details>
<summary><b>💡 Hint</b></summary>

Loop over epochs and over examples; only mistakes change $w$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

For linearly separable data the perceptron convergence theorem guarantees that the rule stops making mistakes after finitely many updates,
so both reach 100 % training accuracy. On non-separable data (e.g. XOR) it never converges.

```python
X_p, y_p = make_blobs(n_samples=100, centers=[[-2, -2], [2, 2]], cluster_std=1.0, random_state=0)
t_p = np.where(y_p == 1, 1, -1)

def perceptron_train(X, t, alpha=0.25, epochs=20):
    Xb = np.c_[np.ones(len(X)), X]
    w = np.zeros(Xb.shape[1])
    for _ in range(epochs):
        for x, ti in zip(Xb, t):
            o = 1 if w @ x > 0 else -1
            w += alpha * (ti - o) * x
    return w

w_p = perceptron_train(X_p, t_p)
acc_mine = np.mean(np.where(np.c_[np.ones(len(X_p)), X_p] @ w_p > 0, 1, -1) == t_p)
```

</details>

### Exercise 15 · The learning rate
*💻 Code · ★★☆*

Fit a single linear unit $o = w_0 + w_1x$ to the data below by batch gradient descent on the **mean** squared error for 50 steps from $w = 0$,
for $\alpha \in \{0.01, 0.1, 0.5, 1.5\}$. Store the final MSE for each $\alpha$ in a dict `final_mse`. Which learning rates are too small,
good, and too large?

In [ ]:
x_lr = rng.normal(size=100)
y_lr = 1.0 + 2.0 * x_lr + 0.1 * rng.normal(size=100)
final_mse = None   # {0.01: ..., 0.1: ..., 0.5: ..., 1.5: ...}

if final_mse is not None:
    _start = np.mean(y_lr ** 2)
    check('alpha=1.5 diverges', final_mse[1.5] > _start, True)
    check('alpha=0.5 converges', final_mse[0.5] < 0.02, True)
    check('alpha=0.01 is still far from the optimum', final_mse[0.01] > 0.1, True)
else:
    check('alpha=1.5 diverges', None, True)

<details>
<summary><b>💡 Hint</b></summary>

Gradient of the MSE: $\frac2N X_b^\top(X_bw - y)$. With standard-normal $x$ the curvature is about 2, so steps above $\approx 1$ overshoot.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$\alpha = 0.01$ moves far too slowly (MSE ≈ 0.53 after 50 steps, versus ≈ 0.009 at the optimum), $0.1$ gets close, $0.5$ converges almost immediately,
and $1.5$ overshoots further on every step and diverges. The learning rate is the most important hyperparameter of gradient descent.

```python
x_lr = rng.normal(size=100)
y_lr = 1.0 + 2.0 * x_lr + 0.1 * rng.normal(size=100)
Xb = np.c_[np.ones(100), x_lr]
final_mse = {}
for a in (0.01, 0.1, 0.5, 1.5):
    w = np.zeros(2)
    for _ in range(50):
        w -= a * 2 / 100 * Xb.T @ (Xb @ w - y_lr)
    final_mse[a] = np.mean((Xb @ w - y_lr) ** 2)
print(final_mse)
```

</details>

### Exercise 16 · Hidden layers beat a linear classifier
*💻 Code · ★★☆*

On `make_moons(n_samples=500, noise=0.2, random_state=0)` with a 70/30 split (`random_state=0`), compare the test accuracy of
`LogisticRegression()` with `MLPClassifier(hidden_layer_sizes=(16,), learning_rate_init=0.01, max_iter=3000, random_state=0)`. Store them in `acc_lin` and `acc_mlp`.
Then try `activation='identity'` for the MLP: what happens and why?

In [ ]:
acc_lin = None
acc_mlp = None
acc_mlp_identity = None

if acc_mlp is not None:
    check('MLP beats logistic regression', acc_mlp > acc_lin + 0.05, True)
    check('identity MLP is no better than linear', acc_mlp_identity <= acc_lin + 0.02, True)
else:
    check('MLP beats logistic regression', None, True)

<details>
<summary><b>💡 Hint</b></summary>

Fit each model on the training part and call `.score(X_test, y_test)`.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Logistic regression draws a straight line (0.84 test accuracy); the ReLU MLP bends the boundary around the moons (about 0.95).
With identity activations the MLP collapses to a linear model (Exercise 2) and performs like logistic regression.

```python
Xm, ym = make_moons(n_samples=500, noise=0.2, random_state=0)
Xtr, Xte, ytr, yte = train_test_split(Xm, ym, test_size=0.3, random_state=0)
acc_lin = LogisticRegression().fit(Xtr, ytr).score(Xte, yte)
acc_mlp = MLPClassifier(hidden_layer_sizes=(16,), learning_rate_init=0.01, max_iter=3000, random_state=0).fit(Xtr, ytr).score(Xte, yte)
acc_mlp_identity = MLPClassifier(hidden_layer_sizes=(16,), activation='identity', max_iter=3000, random_state=0).fit(Xtr, ytr).score(Xte, yte)
print(acc_lin, acc_mlp, acc_mlp_identity)
```

</details>

---
*Done? Revisit the exercises you needed hints for in a few days — retrieval beats rereading. Back to [Noter små](../Noter%20små.md).*